In [1]:
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName("SparkDataFrameDemo").getOrCreate()

# PySpark DataFrame Basics

In [2]:
#help(spark.createDataFrame)

### from tuples (works also from dict)

In [20]:
data = [("Alice", 21), ("Bob", 84), ("Kathy", 41), ]
columns = ["name", "age"]

# create DataFrame by inferring schema
df = df_name_age = spark.createDataFrame(data, columns)
df.show()

+-----+---+
| name|age|
+-----+---+
|Alice| 21|
|  Bob| 84|
|Kathy| 41|
+-----+---+



In [4]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)



In [5]:
df = spark.createDataFrame(data)
df.show()
df.printSchema()

+-----+---+
|   _1| _2|
+-----+---+
|Alice| 21|
|  Bob| 84|
|Kathy| 41|
+-----+---+

root
 |-- _1: string (nullable = true)
 |-- _2: long (nullable = true)



### from an RDD

In [6]:
sc = spark.sparkContext
rdd = sc.parallelize(data)

In [7]:
df_from_rdd = rdd.toDF()
df_from_rdd.show()

+-----+---+
|   _1| _2|
+-----+---+
|Alice| 21|
|  Bob| 84|
|Kathy| 41|
+-----+---+



# Defining a Schema

In [8]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
])

In [9]:
df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()

+-----+---+
| name|age|
+-----+---+
|Alice| 21|
|  Bob| 84|
|Kathy| 41|
+-----+---+

root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)



# Different ways to select columns

In [10]:
import pyspark.sql.functions as F
f = fs = F

In [21]:
data = [("Order_1", 100, 10), ("Order_2", 200, 20)]
columns = ["Order ID", "Price", "Tax"]

df = df_orders = spark.createDataFrame(data, columns)

In [12]:
# this will of course fail
df.Order ID

SyntaxError: invalid syntax (1534644767.py, line 2)

In [13]:
df.select(f.col("Order ID")).show()

+--------+
|Order ID|
+--------+
| Order_1|
| Order_2|
+--------+



In [14]:
df.select(df['Order ID']).show()

+--------+
|Order ID|
+--------+
| Order_1|
| Order_2|
+--------+



In [15]:
_df = (
    df.withColumn("Total", df.Price + df.Tax)
)
_df.show()

+--------+-----+---+-----+
|Order ID|Price|Tax|Total|
+--------+-----+---+-----+
| Order_1|  100| 10|  110|
| Order_2|  200| 20|  220|
+--------+-----+---+-----+



In [16]:
# does not work like this, because df is not modified in-place
_df = (
    df.withColumn("Total", f.col("Price") + f.col("Tax"))
      .withColumn("Discount", df.Total * 0.1)
)
_df.show()

AttributeError: 'DataFrame' object has no attribute 'Total'

In [17]:
_df = (
    df.withColumn("Total", f.col("Price") + f.col("Tax"))
      .withColumn("Discount", f.col("Total") * 0.1)
)
_df.show()

+--------+-----+---+-----+--------+
|Order ID|Price|Tax|Total|Discount|
+--------+-----+---+-----+--------+
| Order_1|  100| 10|  110|    11.0|
| Order_2|  200| 20|  220|    22.0|
+--------+-----+---+-----+--------+



# DataFrame Transformations

In [22]:
df = df_name_age
df.select("name").show()  # select a column

+-----+
| name|
+-----+
|Alice|
|  Bob|
|Kathy|
+-----+



In [24]:
df.filter(df.age > 70).show()

+----+---+
|name|age|
+----+---+
| Bob| 84|
+----+---+



In [25]:
from pyspark.sql.functions import col, when

In [26]:
df.filter((col('age') < 40) & (col('name').startswith('A'))).show()

+-----+---+
| name|age|
+-----+---+
|Alice| 21|
+-----+---+



In [56]:
df.filter((col('age') < 80) & (col('name') == "Kathy")).show()

+-----+---+
| name|age|
+-----+---+
|Kathy| 41|
+-----+---+



In [72]:
df_with_age = (
    df.withColumn("age_bucket",
                  F.when(col("age") < 30, "youngster")
                   .when(col("age") < 60, "mid-citizen")
                   .otherwise("senior-citizen")
    ).withColumn("updated-age", col("age") + 5)
)
df_with_age.show()
                  

+-----+---+--------------+-----------+
| name|age|    age_bucket|updated-age|
+-----+---+--------------+-----------+
|Alice| 21|     youngster|         26|
|  Bob| 84|senior-citizen|         89|
|Kathy| 41|   mid-citizen|         46|
+-----+---+--------------+-----------+



In [63]:
df.show()

+-----+---+
| name|age|
+-----+---+
|Alice| 21|
|  Bob| 84|
|Kathy| 41|
+-----+---+



## Join

In [220]:
df1 = spark.createDataFrame(
    [("maths", 1), ("physics", 2), ],
    ["subject", "id", ],
)
df2 = spark.createDataFrame(
    [("Alice", 1), ("Bob", 2), ("Kathy", 1), ],
    ["name", "id"],
)
df_join = df1.join(df2, "id")
df_join.show()

+---+-------+-----+
| id|subject| name|
+---+-------+-----+
|  1|  maths|Alice|
|  2|physics|  Bob|
|  1|  maths|Kathy|
+---+-------+-----+



In [226]:
df1.explain()

== Physical Plan ==
*(1) Scan ExistingRDD[subject#3216,id#3217L]




In [225]:
df2.explain()

== Physical Plan ==
*(1) Scan ExistingRDD[name#3220,id#3221L]




### The join uses the `SortMergeJoin` which 

1. re-distribute both dataframes across nodes by the join key(s)
   * uses `hash(join_key) % num_partitions` to split data to nodes
   * this uses disk write, network transfer, disk read -> 
1. sorts records in partition
2. merge phase (merge sort)

works well for big data not fitting in memory

In [224]:
df_join.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id#3217L, subject#3216, name#3220]
   +- SortMergeJoin [id#3217L], [id#3221L], Inner
      :- Sort [id#3217L ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(id#3217L, 6), ENSURE_REQUIREMENTS, [plan_id=3920]
      :     +- Filter isnotnull(id#3217L)
      :        +- Scan ExistingRDD[subject#3216,id#3217L]
      +- Sort [id#3221L ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(id#3221L, 6), ENSURE_REQUIREMENTS, [plan_id=3921]
            +- Filter isnotnull(id#3221L)
               +- Scan ExistingRDD[name#3220,id#3221L]




In [223]:
df_join.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- SortMergeJoin Inner (9)
      :- Sort (4)
      :  +- Exchange (3)
      :     +- Filter (2)
      :        +- Scan ExistingRDD (1)
      +- Sort (8)
         +- Exchange (7)
            +- Filter (6)
               +- Scan ExistingRDD (5)


(1) Scan ExistingRDD
Output [2]: [subject#3216, id#3217L]
Arguments: [subject#3216, id#3217L], MapPartitionsRDD[757] at applySchemaToPythonRDD at <unknown>:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [2]: [subject#3216, id#3217L]
Condition : isnotnull(id#3217L)

(3) Exchange
Input [2]: [subject#3216, id#3217L]
Arguments: hashpartitioning(id#3217L, 6), ENSURE_REQUIREMENTS, [plan_id=3920]

(4) Sort
Input [2]: [subject#3216, id#3217L]
Arguments: [id#3217L ASC NULLS FIRST], false, 0

(5) Scan ExistingRDD
Output [2]: [name#3220, id#3221L]
Arguments: [name#3220, id#3221L], MapPartitionsRDD[762] at applySchemaToPythonRDD at <unknown>:0, ExistingRDD, UnknownPartitioning(0)

(6) F

### `BroadcastHashJoin` can be used to avoid these expensive steps for small DataFrames

* broadcasted DataFrame is moved to every node, which avoids the expensive re-distribution

In [118]:
# broadcasting -> send to all the nodes
# use broadcasting for small dataframes
df_broadcast = df1.join( F.broadcast(df2), "id", "inner", )
df_broadcast.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id#2300L, subject#2299, name#2303]
   +- BroadcastHashJoin [id#2300L], [id#2304L], Inner, BuildRight, false
      :- Filter isnotnull(id#2300L)
      :  +- Scan ExistingRDD[subject#2299,id#2300L]
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[1, bigint, false]),false), [plan_id=2246]
         +- Filter isnotnull(id#2304L)
            +- Scan ExistingRDD[name#2303,id#2304L]




## Aliasing

In [74]:
df1.alias("a").join(df2.alias("b"), "id").filter(F.col("a.subject") == "maths").show()

+---+-------+-----+
| id|subject| name|
+---+-------+-----+
|  1|  maths|Alice|
|  1|  maths|Kathy|
+---+-------+-----+



In [119]:
df1 = spark.createDataFrame(
    [("maths", 1), ("physics", 2), ],
    ["subject", "id", ],
)
df2 = spark.createDataFrame(
    [("Alice", 1), ("Bob", 2), ("Kathy", 1), ],
    ["name", "user_id"],
)
df_join = df1.alias("a").join(
    df2.alias("b"), [F.col('a.id') == F.col('b.user_id')],
    'inner',
)
df_join.show()

+-------+---+-----+-------+
|subject| id| name|user_id|
+-------+---+-----+-------+
|  maths|  1|Alice|      1|
|  maths|  1|Kathy|      1|
|physics|  2|  Bob|      2|
+-------+---+-----+-------+



In [120]:
# trying to use the col() function in case of duplicate column

_df1 = spark.createDataFrame(
    [("maths", 1, 90), ("physics", 2, 91), ],
    ["subject", "id", "user_id"],
)
_df2 = spark.createDataFrame(
    [("Alice", 1, 17), ("Bob", 2, 16), ("Kathy", 1, 20), ],
    ["name", "user_id", "id"],
)
_df1.join(
    _df2, [col('id') == col('user_id') ],
    'inner',
).show()

AnalysisException: [AMBIGUOUS_REFERENCE] Reference `id` is ambiguous, could be: [`id`, `id`].

# Grouping and aggregating

In [155]:
df = df_employee = spark.read.csv("employee-dataset/employee-data.csv", header=True, inferSchema=True)
df.show()

+-------+---+------+----------+
|   name|age|salary|department|
+-------+---+------+----------+
|  Alice| 25|  5000|        HR|
|    Bob| 30|  6000|        IT|
|Charlie| 35|  7000|   Finance|
|  David| 28|  5500|     Sales|
|    Eve| 32|  6200|        IT|
|  Frank| 40|  7500|   Finance|
|  Grace| 27|  5300|        HR|
|   Hank| 33|  6400|        IT|
|    Ivy| 29|  5700|     Sales|
|   Jack| 38|  7200|   Finance|
|  Kelly| 26|  5100|        HR|
|   Liam| 31|  5900|        IT|
|    Mia| 34|  6800|   Finance|
|   Noah| 27|  5400|     Sales|
| Olivia| 36|  7100|        IT|
|  Peter| 29|  5600|        HR|
|  Quinn| 37|  7300|   Finance|
|   Rose| 28|  5500|     Sales|
|    Sam| 33|  6500|        IT|
|   Tina| 35|  7000|   Finance|
+-------+---+------+----------+
only showing top 20 rows



In [90]:
kpi1 = df.groupBy("department").agg(F.sum("salary").alias("total_dept_salary"))
kpi1.show()

+----------+-----------------+
|department|total_dept_salary|
+----------+-----------------+
|     Sales|            65000|
|        HR|            60400|
|   Finance|            86400|
|        IT|            91100|
+----------+-----------------+



In [107]:
kpi2 = df.groupBy("department").agg(
    F.count("*").alias("total_std"),
    F.avg("salary").alias("avg_salary"),
    F.max("salary").alias("max_salary"),
    F.min("salary").alias("min_salary"),
    F.mean("salary").alias("mean_salary"),
    F.std("salary").alias("std_salary"),
    F.expr("percentile(salary, 0.99)").alias("perce_099"),
    F.expr("percentile(salary, 0.01)").alias("perce_001"),
)
kpi2.show()

+----------+---------+-----------------+----------+----------+-----------------+-----------------+-----------------+---------+
|department|total_std|       avg_salary|max_salary|min_salary|      mean_salary|       std_salary|        perce_099|perce_001|
+----------+---------+-----------------+----------+----------+-----------------+-----------------+-----------------+---------+
|     Sales|       11|5909.090909090909|      8000|      5300|5909.090909090909|882.5582648806201|           7920.0|   5300.0|
|        HR|       11|5490.909090909091|      6800|      5000|5490.909090909091|531.8919917700312|6720.000000000001|   5010.0|
|   Finance|       13|6646.153846153846|      7500|      5500|6646.153846153846|674.0615508682974|           7488.0|   5500.0|
|        IT|       14|6507.142857142857|      7100|      5900|6507.142857142857|479.5258647300007|           7100.0|   5913.0|
+----------+---------+-----------------+----------+----------+-----------------+-----------------+-------------

In [108]:
kpi2 = df.groupBy("department").agg(
    F.count("*").alias("total_std"),
    F.round(F.avg("salary"), 2).alias("avg_salary"),
    F.max("salary").alias("max_salary"),
    F.min("salary").alias("min_salary"),
    F.round(F.mean("salary"), 2).alias("mean_salary"),
    F.round(F.std("salary"), 2).alias("std_salary"),
    F.round(F.expr("percentile(salary, 0.99)"), 2).alias("perce_099"),
    F.round(F.expr("percentile(salary, 0.01)"), 2).alias("perce_001"),
)
kpi2.show()

+----------+---------+----------+----------+----------+-----------+----------+---------+---------+
|department|total_std|avg_salary|max_salary|min_salary|mean_salary|std_salary|perce_099|perce_001|
+----------+---------+----------+----------+----------+-----------+----------+---------+---------+
|     Sales|       11|   5909.09|      8000|      5300|    5909.09|    882.56|   7920.0|   5300.0|
|        HR|       11|   5490.91|      6800|      5000|    5490.91|    531.89|   6720.0|   5010.0|
|   Finance|       13|   6646.15|      7500|      5500|    6646.15|    674.06|   7488.0|   5500.0|
|        IT|       14|   6507.14|      7100|      5900|    6507.14|    479.53|   7100.0|   5913.0|
+----------+---------+----------+----------+----------+-----------+----------+---------+---------+



In [110]:
kpi2.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- ObjectHashAggregate(keys=[department#634], functions=[count(1), avg(salary#633), max(salary#633), min(salary#633), std(cast(salary#633 as double)), percentile(salary#633, 0.99, 1, 0, 0, false), percentile(salary#633, 0.01, 1, 0, 0, false)])
   +- Exchange hashpartitioning(department#634, 200), ENSURE_REQUIREMENTS, [plan_id=1838]
      +- ObjectHashAggregate(keys=[department#634], functions=[partial_count(1), partial_avg(salary#633), partial_max(salary#633), partial_min(salary#633), partial_std(cast(salary#633 as double)), partial_percentile(salary#633, 0.99, 1, 0, 0, false), partial_percentile(salary#633, 0.01, 1, 0, 0, false)])
         +- FileScan csv [salary#633,department#634] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/employee-dataset/employee-data.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<salary:int,department:string>




# Window functions

In [156]:
df = df_employee
df.take(5)

[Row(name='Alice', age=25, salary=5000, department='HR'),
 Row(name='Bob', age=30, salary=6000, department='IT'),
 Row(name='Charlie', age=35, salary=7000, department='Finance'),
 Row(name='David', age=28, salary=5500, department='Sales'),
 Row(name='Eve', age=32, salary=6200, department='IT')]

In [147]:
from pyspark.sql.window import Window
w = Window

In [158]:
# Create a window function equivalent to the SQL statement in the OVER part like e.g.
# OVER (PARTITION BY department  ORDER BY  salary  DESC)
window_obj = Window.partitionBy("department").orderBy(F.col("salary").desc())

ranked = (
    df.select(
        "name",
        "department",
        "salary",
        F.dense_rank().over(window_obj).alias('dense_rank')
    )
)
ranked.show()

+-------+----------+------+----------+
|   name|department|salary|dense_rank|
+-------+----------+------+----------+
|  Frank|   Finance|  7500|         1|
|  Wendy|   Finance|  7400|         2|
|  Quinn|   Finance|  7300|         3|
|   Jack|   Finance|  7200|         4|
|Charlie|   Finance|  7000|         5|
|   Tina|   Finance|  7000|         5|
|    Mia|   Finance|  6800|         6|
|  Ethan|   Finance|  6400|         7|
| Quincy|   Finance|  6400|         7|
|  Isaac|   Finance|  6200|         8|
|Ulysses|   Finance|  6200|         8|
|  Aaron|   Finance|  5500|         9|
|  Mason|   Finance|  5500|         9|
|  Laura|        HR|  6800|         1|
|   Yara|        HR|  6000|         2|
|  Peter|        HR|  5600|         3|
|  Daisy|        HR|  5600|         3|
|  Paula|        HR|  5600|         3|
|  Grace|        HR|  5300|         4|
|    Uma|        HR|  5200|         5|
+-------+----------+------+----------+
only showing top 20 rows



In [160]:
ranked.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Window [dense_rank(salary#2519) windowspecdefinition(department#2520, salary#2519 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS dense_rank#2586], [department#2520], [salary#2519 DESC NULLS LAST]
   +- Sort [department#2520 ASC NULLS FIRST, salary#2519 DESC NULLS LAST], false, 0
      +- Exchange hashpartitioning(department#2520, 200), ENSURE_REQUIREMENTS, [plan_id=2604]
         +- Project [name#2517, department#2520, salary#2519]
            +- FileScan csv [name#2517,salary#2519,department#2520] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/employee-dataset/employee-data.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<name:string,salary:int,department:string>




In [159]:
ranked.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (6)
+- Window (5)
   +- Sort (4)
      +- Exchange (3)
         +- Project (2)
            +- Scan csv  (1)


(1) Scan csv 
Output [3]: [name#2517, salary#2519, department#2520]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/employee-dataset/employee-data.csv]
ReadSchema: struct<name:string,salary:int,department:string>

(2) Project
Output [3]: [name#2517, department#2520, salary#2519]
Input [3]: [name#2517, salary#2519, department#2520]

(3) Exchange
Input [3]: [name#2517, department#2520, salary#2519]
Arguments: hashpartitioning(department#2520, 200), ENSURE_REQUIREMENTS, [plan_id=2604]

(4) Sort
Input [3]: [name#2517, department#2520, salary#2519]
Arguments: [department#2520 ASC NULLS FIRST, salary#2519 DESC NULLS LAST], false, 0

(5) Window
Input [3]: [name#2517, department#2520, salary#2519]
Arguments: [dense_rank(salary#2519) windowspecdefinition(department#2520, salary#2519 DESC NULLS LAST, specifiedwindowframe(RowFrame, 

# User defined functions

In [165]:
# add salary increments

@F.udf("int")
def int_udf(x):
    return int(1.1 * x)  # type cast required to match F.udf("int")

@F.udf("float")
def float_udf(x):
    return 1.1 * x

df_udf = (
    ranked
    .withColumn("incr_int", int_udf(F.col("salary")))
    .withColumn("incr_float", float_udf(F.col("salary")))
)
df_udf.show()

+-------+----------+------+----------+--------+----------+
|   name|department|salary|dense_rank|incr_int|incr_float|
+-------+----------+------+----------+--------+----------+
|  Frank|   Finance|  7500|         1|    8250|    8250.0|
|  Wendy|   Finance|  7400|         2|    8140|    8140.0|
|  Quinn|   Finance|  7300|         3|    8030|    8030.0|
|   Jack|   Finance|  7200|         4|    7920|    7920.0|
|Charlie|   Finance|  7000|         5|    7700|    7700.0|
|   Tina|   Finance|  7000|         5|    7700|    7700.0|
|    Mia|   Finance|  6800|         6|    7480|    7480.0|
|  Ethan|   Finance|  6400|         7|    7040|    7040.0|
| Quincy|   Finance|  6400|         7|    7040|    7040.0|
|  Isaac|   Finance|  6200|         8|    6820|    6820.0|
|Ulysses|   Finance|  6200|         8|    6820|    6820.0|
|  Aaron|   Finance|  5500|         9|    6050|    6050.0|
|  Mason|   Finance|  5500|         9|    6050|    6050.0|
|  Laura|        HR|  6800|         1|    7480|    7480.